# Intro Lab 2: Using an LLM through an OpenAI-Compatible API

In the first notebook, the model ran locally.

This time, we use the `openai` Python library to talk to an **OpenAI-compatible endpoint**.

The endpoint does **not** have to serve a GPT model. It could serve, for example:

- Qwen
- Llama
- Mistral
- another local or hosted model

The important part is that the server exposes an OpenAI-compatible API.

We will:

1. Configure the endpoint and model.
2. Ask a normal question.
3. Parse JSON.
4. Give the model a simple tool.
5. Build a tiny agent.

## 1. Install the library

In [ ]:
%pip install -q -U openai

## 2. Configure the endpoint

Change these values to match your server.

Example local endpoints:

```text
http://localhost:8000/v1
http://localhost:11434/v1
```

The model name depends on what the server exposes.

In [ ]:
API_BASE_URL = "https://chat-1.ki-awz.iisys.de/api"
API_KEY = "YOUR-API-KEY"
MODEL = "lisa-pro"

## 3. Create the client

The same Python library can talk to different OpenAI-compatible servers by changing `base_url`.

In [ ]:
from openai import OpenAI

client = OpenAI(
    base_url=API_BASE_URL,
    api_key=API_KEY
)

## 4. Send a simple request

We use the normal chat completions interface.

In [ ]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": "Explain what an API is in two short sentences."
        }
    ]
)

print(response.choices[0].message.content)

## 5. Create a small helper function

This gives us the same simple `ask()` pattern as in the first notebook.

In [ ]:
def ask(prompt):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    return response.choices[0].message.content


print(ask("Give me three uses of Python."))

# Part 2: A Tiny Agent

Now we let the model decide whether it wants to use a tool.

We use one simple Python function:

```python
get_temperature(city)
```

The values are fake demo data so that we do not need another API.

## 7. Create the Python tool

In [ ]:
def get_temperature(city):
    temperatures = {
        "Hof": 12,
        "Munich": 16,
        "Berlin": 15
    }

    return temperatures.get(city, "Unknown city")


print(get_temperature("Hof"))

## 8. Describe the tool

OpenAI-compatible APIs can expose function/tool calling.

We tell the model:

- the tool name
- what it does
- which arguments it accepts

In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_temperature",
            "description": "Get the demo temperature for a city.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "Name of the city"
                    }
                },
                "required": ["city"]
            }
        }
    }
]

## 9. Let the model choose whether to call the tool

The model can now either:

- answer normally, or
- request a tool call.

In [ ]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": "What is the temperature in Hof?"
        }
    ],
    tools=tools,
    tool_choice="auto"
)

message = response.choices[0].message

print(message)

## 10. Execute the requested tool

The model only requests the tool.

Our Python program decides what is actually executed.

In [ ]:
if message.tool_calls:
    tool_call = message.tool_calls[0]

    function_name = tool_call.function.name
    arguments = json.loads(tool_call.function.arguments)

    if function_name == "get_temperature":
        tool_result = get_temperature(arguments["city"])

        print("Tool:", function_name)
        print("Arguments:", arguments)
        print("Result:", tool_result)

## 11. Send the tool result back

Now we continue the conversation and give the model the tool result.

In [ ]:
messages = [
    {
        "role": "user",
        "content": "What is the temperature in Hof?"
    },
    message,
    {
        "role": "tool",
        "tool_call_id": tool_call.id,
        "content": str(tool_result)
    }
]

final_response = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    tools=tools
)

print(final_response.choices[0].message.content)

# 12. Put everything into one small agent function

Now we combine the steps.

In [ ]:
def agent(question):
    messages = [
        {
            "role": "user",
            "content": question
        }
    ]

    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=tools,
        tool_choice="auto"
    )

    message = response.choices[0].message

    # No tool needed
    if not message.tool_calls:
        return message.content

    messages.append(message)

    # Execute requested tools
    for tool_call in message.tool_calls:
        arguments = json.loads(tool_call.function.arguments)

        if tool_call.function.name == "get_temperature":
            result = get_temperature(arguments["city"])

            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": str(result)
            })

    # Let the model create the final answer
    final_response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=tools
    )

    return final_response.choices[0].message.content

## 13. Try the agent

In [ ]:
print(agent("What is the temperature in Munich?"))

In [ ]:
print(agent("Explain what Python is in one sentence."))

## 14. Optional exercise

Add another function:

```python
def get_population(city):
    ...
```

Then add it to `tools`.

Try asking:

```text
What is the population of Munich?
```

or:

```text
Give me the temperature and population of Berlin.
```